# Self-attention with trainable weights

This notebook builds **single-head, unmasked self-attention** in two ways:
with explicit parameter matrices and with PyTorch linear layers.
Both implementations compute the same operation when their weights match.

The example follows [Chapter 3 of LLMs from Scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb).

We process one sequence at a time, without a batch dimension. Every token can
attend to every token, including itself and later tokens. A GPT language model
also needs a **causal mask** to prevent access to future tokens; that is outside
this example. The weights here are trainable, but remain randomly initialized
because we do not run a training loop.

> **Origin and license.** This notebook is derived from [`ch03/01_main-chapter-code/ch03.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb) in the repository [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Copyright 2023-2026 Sebastian Raschka, licensed under the [Apache License 2.0](../LICENSES/Apache-2.0-LLMs-from-scratch.txt).
>
> **Modified by L. Saetta (2026):** explanations rewritten and extended, and code adapted to this repository. See [NOTICE](../NOTICE).

In [1]:
import torch
from torch import nn

## 1. From input embeddings to queries, keys, and values

Each row of the input matrix $X$ represents one token. Three learned projections
produce different representations of that same sequence:

- A **query** describes what a token is looking for.
- A **key** describes what a token can be matched against.
- A **value** carries the information that will be combined into the output.

These are useful intuitions; the projections acquire their meaning through
training rather than through hand-written rules. This is *self*-attention
because queries, keys, and values all come from the same input sequence.

$$Q = XW_Q, \qquad K = XW_K, \qquad V = XW_V$$

We use the same output dimension for all three projections:

| Tensor | Shape | Meaning |
| --- | --- | --- |
| Input $X$ | `(num_tokens, d_in)` | One input embedding per token |
| Each projection matrix | `(d_in, d_out)` | Shared across all token positions |
| Queries $Q$, keys $K$, values $V$ | `(num_tokens, d_out)` | Projected token representations |
| Attention scores $QK^T$ | `(num_tokens, num_tokens)` | One score for each query–key pair |
| Attention weights $A$ | `(num_tokens, num_tokens)` | One distribution over keys per query |
| Context vectors $Z = AV$ | `(num_tokens, d_out)` | One output vector per token |

$$A = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right),
\qquad Z = AV, \qquad d_k = d_{out}$$

The square-root scaling helps keep dot products from growing too large as the
key dimension increases. Softmax acts along the **last axis**: for each query,
the weights over all keys sum to one. The output is a weighted sum of the
**values**, not of the original input embeddings.

In [2]:
class SelfAttention_v1(nn.Module):
    """Single-head self-attention with explicit projection matrices.

    Args:
        d_in: Number of features in each input token embedding.
        d_out: Number of features in each query, key, value, and output vector.
    """

    def __init__(self, d_in: int, d_out: int) -> None:
        super().__init__()
        # nn.Parameter registers these tensors as trainable model parameters.
        # One matrix is shared by every token; there is no matrix per position.
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute one context vector per token in an unbatched sequence.

        Args:
            x: Input embeddings with shape (num_tokens, d_in).

        Returns:
            Context vectors with shape (num_tokens, d_out).
        """
        # Each projection has shape (num_tokens, d_out).
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        # Row i compares query i with every key: (num_tokens, num_tokens).
        # .T is appropriate here because keys is a two-dimensional matrix.
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        # Each row is a weighted sum of values: (num_tokens, d_out).
        context_vectors = attn_weights @ values
        return context_vectors

## 2. Run a small example

The six rows below are illustrative input embeddings for “Your journey starts
with one step”. They are supplied directly, rather than obtained from a tokenizer
and an embedding layer. The phrase labels help us track token positions.

Here `d_in = d_out = 3`, but the projection can also change the feature dimension.
The seed makes the initial weights reproducible in this example.

In [3]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your
        [0.55, 0.87, 0.66],  # journey
        [0.57, 0.85, 0.64],  # starts
        [0.22, 0.58, 0.33],  # with
        [0.77, 0.25, 0.10],  # one
        [0.05, 0.80, 0.55],  # step
    ]
)

d_in = inputs.shape[-1]
d_out = 3

torch.manual_seed(123)
attention_v1 = SelfAttention_v1(d_in, d_out)
outputs_v1 = attention_v1(inputs)

print("Input shape:", tuple(inputs.shape))
print("Output shape:", tuple(outputs_v1.shape))
outputs_v1

Input shape: (6, 3)
Output shape: (6, 3)


tensor([[0.6692, 1.0276, 1.1106],
        [0.6864, 1.0577, 1.1389],
        [0.6860, 1.0570, 1.1383],
        [0.6738, 1.0361, 1.1180],
        [0.6711, 1.0307, 1.1139],
        [0.6783, 1.0441, 1.1252]], grad_fn=<MmBackward0>)

## 3. Inspect the intermediate tensors

For this six-token example, the score matrix is `(6, 6)`. Its row index selects
the query token and its column index selects the key token. Softmax turns each
row into a distribution over the six values. This inspection uses `no_grad()`
because we only want to display the calculation, without building a gradient graph.

In [4]:
with torch.no_grad():
    queries = inputs @ attention_v1.W_query
    keys = inputs @ attention_v1.W_key
    values = inputs @ attention_v1.W_value
    scores = queries @ keys.T
    weights = torch.softmax(scores / keys.shape[-1] ** 0.5, dim=-1)
    context_vectors = weights @ values

print("Q, K, V shapes:", tuple(queries.shape), tuple(keys.shape), tuple(values.shape))
print("Score shape:", tuple(scores.shape))
print("Context vector shape:", tuple(context_vectors.shape))
print("Sum of attention weights for each query:", weights.sum(dim=-1))

Q, K, V shapes: (6, 3) (6, 3) (6, 3)
Score shape: (6, 6)
Context vector shape: (6, 3)
Sum of attention weights for each query: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 4. Express the projections with linear layers

`nn.Linear(d_in, d_out, bias=False)` applies the same kind of projection as
`x @ W`. PyTorch stores its weight matrix as `(d_out, d_in)` and computes
`x @ weight.T`. The first implementation stores its matrices as `(d_in, d_out)`.

Setting `qkv_bias=True` adds a learned bias to each projection. We leave biases
disabled to match the first implementation.

The two versions use different random initialization schemes: `torch.rand`
samples from `[0, 1)`, while `nn.Linear` initializes its weights differently.
Resetting the seed therefore does **not** make their initial outputs equal.

In [5]:
class SelfAttention_v2(nn.Module):
    """Single-head self-attention with linear query, key, and value projections.

    Args:
        d_in: Number of features in each input token embedding.
        d_out: Number of features in each query, key, value, and output vector.
        qkv_bias: Whether each projection includes a trainable bias.
    """

    def __init__(self, d_in: int, d_out: int, qkv_bias: bool = False) -> None:
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute one context vector per token in an unbatched sequence.

        Args:
            x: Input embeddings with shape (num_tokens, d_in).

        Returns:
            Context vectors with shape (num_tokens, d_out).
        """
        # Linear layers project the last dimension from d_in to d_out.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Normalize over keys for each query, just as in the first version.
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        context_vectors = attn_weights @ values
        return context_vectors

In [6]:
torch.manual_seed(123)
attention_v2 = SelfAttention_v2(d_in, d_out)
outputs_v2 = attention_v2(inputs)

print("Output shape:", tuple(outputs_v2.shape))
outputs_v2

Output shape: (6, 3)


tensor([[ 0.2633,  0.4277, -0.1353],
        [ 0.2641,  0.4296, -0.1350],
        [ 0.2641,  0.4296, -0.1350],
        [ 0.2647,  0.4316, -0.1381],
        [ 0.2642,  0.4303, -0.1373],
        [ 0.2648,  0.4316, -0.1375]], grad_fn=<MmBackward0>)

## 5. Check equivalence with matching weights

To compare the operations, copy the first version's projection matrices into
the second version, transposing them to match `nn.Linear` storage. We use a
separate instance to preserve the randomly initialized example above.

`torch.no_grad()` lets us initialize parameters without recording the copies in
the gradient graph. It does not make the parameters permanently non-trainable.

In [7]:
attention_v2_matched = SelfAttention_v2(d_in, d_out)
with torch.no_grad():
    attention_v2_matched.W_query.weight.copy_(attention_v1.W_query.T)
    attention_v2_matched.W_key.weight.copy_(attention_v1.W_key.T)
    attention_v2_matched.W_value.weight.copy_(attention_v1.W_value.T)

matched_outputs = attention_v2_matched(inputs)
print("Matching outputs:", torch.allclose(outputs_v1, matched_outputs))

Matching outputs: True


## What is trainable here?

The three projection matrices are trainable parameters. Queries, keys, values,
and attention weights are **computed from the current input and parameters**;
they are not separate model parameters. During training, a loss is differentiated
through these operations to obtain gradients for the projection matrices.
An optimizer then updates the parameters.

The outputs displayed here still come from random initial weights. A forward
pass alone does not train the model or establish meaningful token relationships.

This example has no causal mask, positional encoding, multiple heads, or final
output projection. It illustrates the core attention calculation, rather than
a complete GPT attention block. It accepts inputs of shape `(num_tokens, d_in)`;
to extend the calculation to batches, replace `keys.T` with
`keys.transpose(-2, -1)` in both classes.